# Tokenizer word, subword, character based.


# Word based tokenizer.
# issues : Out of vocabulary words ? how to handle it.
# Boy and Boys are similar semantically -> but different words...Tokens might not be related. So semantic meaning lost

# Character based tokenizer:
## a, b, c , d -> tokens ...so token size is very small . eg :256 characters.
## Do you thing just 26 unique tokens can capture the whole english language. Will it work?
## meanings associated with words are completely lost.

# Rules
## High frequency word - not split.
## low frequency words/rare word - split.

## Eg: 'The' is not split.
## 'dhiraj' is split in to subwords.
## 'boy' and 'boys' case -> might be 'boy' is not split, but 'boys' split as 'boy' and 's'.
## perfect sense as semantic meaning still intact.

## this apples -> 'apple' and 's'
## this makes the model learn that different sequence of tokens has some meaning.


# benefits of subword tokenization ? - think about it.
# mordernization , substitution , location

# oov problem solved.
# bpe reduces vocabulary size.
# eg : boy, boys, apple , apples -> 4 tokens in word tokenization.
# subword : boy, apple, s -> 3 tokens

# Root words tokens + fillers  tokens => less tokens.
## learn -=> learning, learnt, learned


# BPE - Data compression algorithm.

## Most common word should be represented by one token.
## rare word broken in to subwords - > that might be occuring pretty high number of times.

## https://www.youtube.com/watch?v=APnKbi448O4
## https://sebastianraschka.com/blog/2025/bpe-from-scratch.html : to understand.
## to directly use it as used in GPT family: use the tiktoken library.
## you can load directly the tokenizer used in GPT-2, GPT-4 models.

# Try to build a BPE tokenizer from scratch.
# STEPS:
## 1. Need 2 things : to save token -> id, id -> token dictionary.
## 2. Initially save all single character tokens -> this is bascially responsible for BPEs power to encode every thing as token.
## 3. Now, try to find frequent occuring pairs -> b, o, y ->3 tokens, but if "boy" occurs frequently -> then bo, oy also added -> boy also added.
## 4. This helps in creating things like lowest, nearest -> low, near, est kind of semantically meaningfull filler to root words.
## 5. This is vocabulary.

## 6. Flow for encoding new input scentences.
## 7. Flow to decode tokenized scentence.

In [122]:
from collections import Counter
class Tokenizer():
    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.tokens_to_id = {}
        self.id_to_tokens = {}
        self.maxSizedToken = 1
        # by default for each character add the token.
        for i in range(256):
            token = chr(i)
            self.tokens_to_id[token] = i
            self.id_to_tokens[i] = token
    def create_vocab(self, input_data):
        # find all the unique characters in input , not yet added in vocabulary:
        for character in set(input_data):
            if character not in self.tokens_to_id.keys():
                self.tokens_to_id[character] = len(self.tokens_to_id)
                self.id_to_tokens[len(self.tokens_to_id)] = character
        # now find the best pairs:
        # each combination of characters should be represented as one...so first convert to token_ids.
        # then do the process.
        token_ids = []
        for char in input_data:
            token_ids.append(self.tokens_to_id[char])
        
        # token_ids this is what needs to be worked on.
        while len(self.tokens_to_id) < self.vocab_size:
            print("token ids :", len(token_ids))
            pairs = zip(token_ids[:-1], token_ids[1:])
            counter = Counter(pairs)
            # print(counter)
            most_frequent_pair = max(counter.items(), key= lambda x: x[1])
            
            if most_frequent_pair[1] == 1:
                break
            print("most_frequent_pair : ",most_frequent_pair)
            most_frequent_pair = most_frequent_pair[0]

            # what are the characters that make up this most_frequent_pair:
            new_token = self.id_to_tokens[most_frequent_pair[0]] + self.id_to_tokens[most_frequent_pair[1]] 
            print("combining <", str(self.id_to_tokens[most_frequent_pair[0]]), "> and <", self.id_to_tokens[most_frequent_pair[1]],">" )
            next_free_token_id = len(self.id_to_tokens)
            self.tokens_to_id[new_token] = next_free_token_id
            self.id_to_tokens[next_free_token_id] = new_token
            self.maxSizedToken = max(self.maxSizedToken, len(new_token))
            # go thought the token_id list and find where these 2 tokens occur and replace with a next_free_token_id
            new_token_ids = []
            i = 0
            while i < len(token_ids) - 1:
                if token_ids[i] == most_frequent_pair[0] and token_ids[i+1] == most_frequent_pair[1]:
                   new_token_ids.append(next_free_token_id)
                   i += 2
                else:
                   new_token_ids.append(token_ids[i])
                   i += 1
            if token_ids[-2] != most_frequent_pair[0] or token_ids[-1] != most_frequent_pair[1]:
                new_token_ids.append(token_ids[-1])

            token_ids = new_token_ids[:]      

            # print(len(self.tokens_to_id)
            print(self.id_to_tokens)

    def encode(self, input_to_encode):
        encoded_tokens = []
        cur_position = 0
        while cur_position < len(input_to_encode):
            # print(cur_position)
            endBoundaryOfToken = min(len(input_to_encode) - 1, cur_position + self.maxSizedToken - 1)

            if endBoundaryOfToken == cur_position:
                print("checking ", input_to_encode[cur_position: endBoundaryOfToken + 1])
                print("found")
                encoded_tokens.append(self.tokens_to_id[input_to_encode[cur_position]])
                cur_position += 1
            else:
                while endBoundaryOfToken >= cur_position:
                    print("checking ", input_to_encode[cur_position: endBoundaryOfToken + 1])
                    if input_to_encode[cur_position: endBoundaryOfToken + 1] in self.tokens_to_id:
                        print("found")
                        encoded_tokens.append(self.tokens_to_id[input_to_encode[cur_position: endBoundaryOfToken + 1]])
                        cur_position = endBoundaryOfToken + 1
                        # print("encoded_tokens :", encoded_tokens)
                        break
                    else:
                        endBoundaryOfToken -= 1
        return encoded_tokens
    def decode(self, token_ids):
        return ''.join([ self.id_to_tokens[token] for token in token_ids])





In [123]:
tokeniker = Tokenizer(vocab_size =270)
tokeniker.create_vocab("this is dhirajs laptop, this is good for this price")
encoded_tokens = tokeniker.encode("encode this")

token ids : 51
most_frequent_pair :  ((115, 32), 6)
combining < s > and <   >
{0: '\x00', 1: '\x01', 2: '\x02', 3: '\x03', 4: '\x04', 5: '\x05', 6: '\x06', 7: '\x07', 8: '\x08', 9: '\t', 10: '\n', 11: '\x0b', 12: '\x0c', 13: '\r', 14: '\x0e', 15: '\x0f', 16: '\x10', 17: '\x11', 18: '\x12', 19: '\x13', 20: '\x14', 21: '\x15', 22: '\x16', 23: '\x17', 24: '\x18', 25: '\x19', 26: '\x1a', 27: '\x1b', 28: '\x1c', 29: '\x1d', 30: '\x1e', 31: '\x1f', 32: ' ', 33: '!', 34: '"', 35: '#', 36: '$', 37: '%', 38: '&', 39: "'", 40: '(', 41: ')', 42: '*', 43: '+', 44: ',', 45: '-', 46: '.', 47: '/', 48: '0', 49: '1', 50: '2', 51: '3', 52: '4', 53: '5', 54: '6', 55: '7', 56: '8', 57: '9', 58: ':', 59: ';', 60: '<', 61: '=', 62: '>', 63: '?', 64: '@', 65: 'A', 66: 'B', 67: 'C', 68: 'D', 69: 'E', 70: 'F', 71: 'G', 72: 'H', 73: 'I', 74: 'J', 75: 'K', 76: 'L', 77: 'M', 78: 'N', 79: 'O', 80: 'P', 81: 'Q', 82: 'R', 83: 'S', 84: 'T', 85: 'U', 86: 'V', 87: 'W', 88: 'X', 89: 'Y', 90: 'Z', 91: '[', 92: '\\', 93:

# Use try to decode this encoded scentence.

In [135]:
# print(list(tokeniker.tokens_to_id.keys())[-10:])

encoded_tokens = tokeniker.encode("this dhiraj wants to encode this is .")
output = tokeniker.decode(encoded_tokens)
print("encoded_tokens",encoded_tokens)
print("output:",output)

checking  this dhi
checking  this dh
checking  this d
checking  this 
found
checking  dhiraj w
checking  dhiraj 
checking  dhiraj
checking  dhira
checking  dhir
checking  dhi
checking  dh
checking  d
found
checking  hiraj wa
checking  hiraj w
checking  hiraj 
checking  hiraj
checking  hira
checking  hir
checking  hi
checking  h
found
checking  iraj wan
checking  iraj wa
checking  iraj w
checking  iraj 
checking  iraj
checking  ira
checking  ir
checking  i
found
checking  raj want
checking  raj wan
checking  raj wa
checking  raj w
checking  raj 
checking  raj
checking  ra
checking  r
found
checking  aj wants
checking  aj want
checking  aj wan
checking  aj wa
checking  aj w
checking  aj 
checking  aj
checking  a
found
checking  j wants 
checking  j wants
checking  j want
checking  j wan
checking  j wa
checking  j w
checking  j 
checking  j
found
checking   wants t
checking   wants 
checking   wants
checking   want
checking   wan
checking   wa
checking   w
checking   
found
checking  want

# Challenges
## how is size of vocabulary decided?
### min size of token?
### just a treshold on number of different tokens.
### threshold on minimum number of occurances of a pattern.
### Are lower case upper case different ?
### how to handle multiple spaces, and other things .